# EEG-Flow — Grand average

Loads per-participant evoked files from `derivatives_v2`, computes grand
averages per condition, and saves as pickle files for use by plot notebooks.

**Toggles** — edit the cell below before running:
- `PAPER`: `'load'` or `'flow'`
- `PARADIGM`: `'active'` or `'passive'` (or `'active-n35'` etc. for sub-cohorts)
- `RUN_STIM_RAW`, `RUN_STIM_DIFF`, `RUN_RESP_RAW`: toggle sections
- `RUN_TASKDIFF`: load paper only — computes Simple−Complex within-subject difference
- `TRIALS`: `'correct'` (primary) or `'alltrials'` — which stimulus-locked evoked files to read

CSD and baseline correction are set automatically per paper:
- Flow response-locked: CSD=True, bc-200-0ms
- Load response-locked: CSD=False, bc-200-100ms

Valid combinations for response-locked:
- Flow: `CSD=True` + `BASELINE_CORR='bc-200-0'`
- Load: `CSD=False` + `BASELINE_CORR='bc-200-100'`

Outputs go to `derivatives_v2/outputs/grand_averages/{paper}/`:
- stimulus-locked: `raw/{TRIALS}/`, `diff/{TRIALS}/`, `taskdiff/{TRIALS}/`
- response-locked (hits only, independent of `TRIALS`): `raw/`, `taskdiff/`

Last edit: 30.10.2026 @anguyen

In [31]:
# ==============================================================================
# TOGGLES — edit here
# ==============================================================================
PAPER         = 'flow'    # 'load' or 'flow'
PARADIGM      = 'active'  # 'active', 'passive', or 'active-n35' etc.
RUN_STIM_RAW  = True
RUN_STIM_DIFF = True
RUN_RESP_RAW  = True
RUN_TASKDIFF  = True      # load paper only — Simple minus Complex

TRIALS = 'correct'   # 'correct' (primary) | 'alltrials'

In [32]:
from __future__ import annotations

import pickle
import sys
from datetime import datetime
from pathlib import Path
from sys import platform

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))

import config_flow
import config_load
from mne import combine_evoked, read_evokeds
from mne.evoked import Evoked
from mne_lsl.utils._checks import ensure_path
from mne_lsl.utils.logs import warn
from participants import get_participants
from tqdm import tqdm
from utils import save_pkl

date          = datetime.today().strftime('%Y%m%d')
PARADIGM_REAL = PARADIGM.split('-')[0]
STUDY         = 'Active' if PARADIGM_REAL == 'active' else 'Passive'
DEVIANT_LABEL = 'target' if PARADIGM_REAL == 'active' else 'deviant'

# CSD and baseline set automatically per paper
if PAPER == 'flow':
    BASELINE_CORR = 'bc-200-0'
    CSD           = True
elif PAPER == 'load':
    BASELINE_CORR = 'bc-200-100'
    CSD           = False
label_csd = 'CSD' if CSD else 'NOCSD'

# --- paths ---
if platform == 'win32':
    root = Path(f'L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')
else:
    root = Path(f'/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')

assert TRIALS in ('correct', 'alltrials'), f'Unknown TRIALS: {TRIALS}'

if PARADIGM == PARADIGM_REAL:
    base = root / 'outputs' / 'grand_averages' / PAPER
else:
    cohort = PARADIGM.split('-', 1)[1]
    base = root / 'outputs' / 'grand_averages' / PAPER / '_other_cohorts' / cohort

# stimulus-locked: depend on TRIALS
output_path_raw      = base / 'raw'      / TRIALS
output_path_diff     = base / 'diff'     / TRIALS
output_path_taskdiff = base / 'taskdiff' / TRIALS
# response-locked: hits only, identical for both TRIALS — kept outside the subfolder
output_path_resp          = base / 'raw'
output_path_taskdiff_resp = base / 'taskdiff'

for p in [output_path_raw, output_path_diff, output_path_taskdiff,
          output_path_resp, output_path_taskdiff_resp]:
    p.mkdir(parents=True, exist_ok=True)

PARTICIPANTS = get_participants(PAPER, PARADIGM)

# --- warn on unused combinations ---
if PARADIGM_REAL == 'active' and RUN_RESP_RAW:
    if CSD and BASELINE_CORR == 'bc-200-100':
        warn('CSD=True + bc-200-100 is not a standard combination — '
             'no v2 file exists for this. Use CSD=True + bc-200-0 (flow) '
             'or CSD=False + bc-200-100 (load).')
    if not CSD and BASELINE_CORR == 'bc-200-0':
        warn('CSD=False + bc-200-0 is not a standard combination — '
             'no v2 file exists for this. Use CSD=True + bc-200-0 (flow) '
             'or CSD=False + bc-200-100 (load).')

if RUN_TASKDIFF and (PAPER != 'load' or PARADIGM_REAL != 'active'):
    warn('RUN_TASKDIFF is only meaningful for load paper, active paradigm — skipping.')
    RUN_TASKDIFF = False

print('=' * 60)
print('GRAND AVERAGE CONFIGURATION')
print('=' * 60)
print(f'  Paper:            {PAPER.upper()}')
print(f'  Paradigm:         {PARADIGM_REAL.upper()}')
print(f'  Trials (stim):    {TRIALS}')
print(f'  CSD (resp):       {CSD}')
print(f'  Baseline (resp):  {BASELINE_CORR}')
print(f'  Stim raw:         {RUN_STIM_RAW}')
print(f'  Stim diff:        {RUN_STIM_DIFF}')
print(f'  Resp raw:         {RUN_RESP_RAW}')
print(f'  Task diff:        {RUN_TASKDIFF}')
print(f'  Participants:     {len(PARTICIPANTS)}')
print(f'  Output stim:      {output_path_raw.relative_to(root)}')
print(f'  Output resp:      {output_path_resp.relative_to(root)}')
print('=' * 60)

GRAND AVERAGE CONFIGURATION
  Paper:            FLOW
  Paradigm:         ACTIVE
  CSD (resp):       True
  Baseline (resp):  bc-200-0
  Stim raw:         True
  Stim diff:        True
  Resp raw:         True
  Task diff:        False
  Participants:     36
  Output:           outputs\grand_averages\flow\raw


C:\Users\anguyen\AppData\Local\Temp\ipykernel_9404\291874042.py:74: RuntimeWarning: RUN_TASKDIFF is only meaningful for the load paper — skipping.
  warn('RUN_TASKDIFF is only meaningful for the load paper — skipping.')


In [33]:
# ==============================================================================
# CONDITION MAP
# ==============================================================================

if PAPER == 'load':
    CONDITION_MAP = {
        'simple':  {'task': 'oddball', 'run': None},
        'complex': {'task': 'UT',      'run': None},
    }
    stim_events = ['standard', DEVIANT_LABEL, 'novel']
    diff_events = [f'{DEVIANT_LABEL}-standard', 'novel-standard']
    resp_events = ['response']
else:  # flow
    CONDITION_MAP = {
        'lowflow':  {'task': 'UT', 'run': 'low_mapping'},
        'highflow': {'task': 'UT', 'run': 'high_mapping'},
    }
    stim_events = ['standard', DEVIANT_LABEL, 'novel']
    diff_events = [f'{DEVIANT_LABEL}-standard', 'novel-standard',
                   f'{DEVIANT_LABEL}-novel']
    resp_events = ['response']

In [34]:
# ==============================================================================
# I/O FUNCTIONS
# ==============================================================================

def read_evoked_file(root, uid, gid, task, domain, event, run=None):
    """Read one stimulus- or response-locked evoked from disk."""
    root = ensure_path(root, must_exist=True)
    pid  = f'sub-{uid}-{gid}'

    if domain == 'resp':
        if run is not None:
            # flow: per-run folder
            run_dir = root / pid / f'{pid}_task-{task}_run-{run}'
            if CSD:
                fname = f'{pid}_task-{task}_run-{run}_step8_responselocked_CSD_bc-200-0-ave.fif'
            else:
                fname = f'{pid}_task-{task}_run-{run}_step8_responselocked_bc-200-100-ave.fif'
            fpath = run_dir / fname
        else:
            # load: concatenated folder
            fpath = (root / pid / 'concatenated' /
                     f'{pid}_task-{task}_step8_responselocked_bc-200-100-interp-ave.fif')

        evokeds = read_evokeds(fpath, verbose=False)
        assert len(evokeds) == 1, f'Expected 1 evoked in {fpath.name}, got {len(evokeds)}'
        evoked = evokeds[0]

    else:  # stim
        disk_event = 'target' if event == 'deviant' else event
        if run is not None:
            # flow: per-run file in run subfolder
            run_dir = root / pid / f'{pid}_task-{task}_run-{run}'
            fname   = f'{pid}_task-{task}_run-{run}_step8_stimlocked_{TRIALS}-ave.fif'
            fpath   = run_dir / fname
        else:
            # load: concatenated file
            fpath = root / pid / 'concatenated' / f'{pid}_task-{task}_step8_stimlocked_{TRIALS}-interp-ave.fif'

        evokeds  = read_evokeds(fpath, verbose=False)
        matching = [e for e in evokeds if e.comment == disk_event]
        assert len(matching) == 1, \
            f'Expected 1 evoked for {disk_event} in {fpath.name}, got {len(matching)}'
        evoked = matching[0]

    evoked.comment = uid
    if evoked.info['bads']:
        evoked.interpolate_bads(reset_bads=True, verbose=False)
    return evoked


def generate_differences(subject_map, keys):
    """Compute difference waves for one participant."""
    diffs = {}
    pairs = [
        (f'{DEVIANT_LABEL}-standard', DEVIANT_LABEL, 'standard'),
        ('novel-standard',            'novel',        'standard'),
        (f'{DEVIANT_LABEL}-novel',    DEVIANT_LABEL,  'novel'),
    ]
    for key, a, b in pairs:
        if key in keys and a in subject_map and b in subject_map:
            diff = combine_evoked([subject_map[a], subject_map[b]], weights=[1, -1])
            diff.comment = subject_map[b].comment
            diffs[key] = diff
    return diffs


def pivot_and_save(dataset, target_keys, folder_path, filename_template, log_label):
    """Pivot {uid: {cond: Evoked}} -> {cond: [Evoked,...]} and save pkl."""
    pivoted = {k: [] for k in target_keys}
    for uid_data in dataset.values():
        for cond, evoked in uid_data.items():
            if cond in pivoted and isinstance(evoked, Evoked):
                pivoted[cond].append(evoked)
    if not any(pivoted.values()):
        warn(f'Empty dataset for {log_label} — skipping.')
        return
    sample_size = max(len(v) for v in pivoted.values())
    fname = filename_template.format(
        count=sample_size, date=date, paradigm=PARADIGM
    )
    save_pkl(pivoted, folder_path / fname, log_label)

In [35]:
# ==============================================================================
# DATA HARVESTING
# ==============================================================================

raw_stim_data:  dict = {}
diff_stim_data: dict = {}
raw_resp_data:  dict = {}

for uid, gid in tqdm(PARTICIPANTS.items(), desc='Loading evokeds', unit='sub'):
    raw_stim_data[uid]  = {}
    diff_stim_data[uid] = {}
    raw_resp_data[uid]  = {}

    for cond_label, config in CONDITION_MAP.items():
        task = config['task']
        sub_stim_pool = {}

        # resolve run index (flow only)
        if PAPER == 'flow':
            run_idx = 1 if gid in ['G1', 'G3', 'G5', 'G7'] else 2
            if config['run'] == 'high_mapping':
                run_idx = 2 if gid in ['G1', 'G3', 'G5', 'G7'] else 1
        else:
            run_idx = None

        # stim raw
        if RUN_STIM_RAW:
            for ev in stim_events:
                try:
                    evoked = read_evoked_file(
                        root, uid, gid, task, 'stim', ev, run=run_idx
                    )
                    sub_stim_pool[ev] = evoked
                    raw_stim_data[uid][f'{cond_label}_{ev}'] = evoked
                except FileNotFoundError:
                    warn(f'Missing: uid={uid} task={task} event={ev} run={run_idx}')
                except Exception as e:
                    warn(f'Error: uid={uid} task={task} event={ev} -> {e}')

        # stim diff
        if RUN_STIM_DIFF and sub_stim_pool:
            for diff_key, diff_ev in generate_differences(
                    sub_stim_pool, diff_events).items():
                diff_stim_data[uid][f'{cond_label}_{diff_key}'] = diff_ev

        # response raw (active only)
        if RUN_RESP_RAW and PARADIGM_REAL == 'active':
            try:
                evoked_resp = read_evoked_file(
                    root, uid, gid, task, 'resp', 'response', run=run_idx
                )
                raw_resp_data[uid][f'{cond_label}_response'] = evoked_resp
            except FileNotFoundError:
                warn(f'Missing: uid={uid} task={task} event=response run={run_idx}')
            except Exception as e:
                warn(f'Error: uid={uid} task={task} event=response -> {e}')

Loading evokeds: 100%|████████████████████████████████████████████████████████████████| 36/36 [00:39<00:00,  1.09s/sub]


In [36]:
# ==============================================================================
# SERIALIZATION — per condition
# ==============================================================================

print('\nStructuring and saving datasets...')

for cond_label in CONDITION_MAP:

    if RUN_STIM_RAW:
        pivot_and_save(
            raw_stim_data,
            [f'{cond_label}_{ev}' for ev in stim_events],
            output_path_raw,
            '{date}_{paradigm}_{count}p_cond-' + cond_label + '_stim_raw.pkl',
            f'[{cond_label}] stim raw',
        )

    if RUN_STIM_DIFF:
        pivot_and_save(
            diff_stim_data,
            [f'{cond_label}_{df}' for df in diff_events],
            output_path_diff,
            '{date}_{paradigm}_{count}p_cond-' + cond_label + '_stim_diff.pkl',
            f'[{cond_label}] stim diff',
        )

    if RUN_RESP_RAW and PARADIGM_REAL == 'active':
        pivot_and_save(
            raw_resp_data,
            [f'{cond_label}_response'],
            output_path_resp,
            '{date}_{paradigm}_{count}p_cond-' + cond_label +
            f'_resp_{BASELINE_CORR}_{label_csd}_raw.pkl',
            f'[{cond_label}] resp raw',
        )

print('\nDone.')


Structuring and saving datasets...
Saved [lowflow] stim raw → 20261015_active_36p_cond-lowflow_stim_raw.pkl
Saved [lowflow] stim diff → 20261015_active_36p_cond-lowflow_stim_diff.pkl
Saved [lowflow] resp raw → 20261015_active_36p_cond-lowflow_resp_bc-200-0_CSD_raw.pkl
Saved [highflow] stim raw → 20261015_active_36p_cond-highflow_stim_raw.pkl
Saved [highflow] stim diff → 20261015_active_36p_cond-highflow_stim_diff.pkl
Saved [highflow] resp raw → 20261015_active_36p_cond-highflow_resp_bc-200-0_CSD_raw.pkl

Done.


In [37]:
# ==============================================================================
# TASK DIFFERENCE — load paper only
# Simple minus Complex (oddball minus UT) within-subject, then grand average
# Reads from raw_stim_data already harvested above — no extra file reads
# ==============================================================================

if PAPER == "load" and PARADIGM == "active":
    if RUN_TASKDIFF:
        print('\nComputing task difference (Simple - Complex)...')

        taskdiff_stim_data: dict = {}
        taskdiff_resp_data: dict = {}

        for uid in PARTICIPANTS:
            taskdiff_stim_data[uid] = {}
            taskdiff_resp_data[uid] = {}

            # stim raw: simple - complex per event type
            if RUN_STIM_RAW:
                for ev in stim_events:
                    key_simple  = f'simple_{ev}'
                    key_complex = f'complex_{ev}'
                    if (key_simple  in raw_stim_data[uid] and
                        key_complex in raw_stim_data[uid]):
                        a = raw_stim_data[uid][key_simple].copy().pick('eeg')
                        b = raw_stim_data[uid][key_complex].copy().pick('eeg')
                        diff = combine_evoked([a, b], weights=[1, -1])
                        diff.comment = uid
                        taskdiff_stim_data[uid][ev] = diff

            # stim diff: simple - complex per difference wave
            if RUN_STIM_DIFF:
                for df in diff_events:
                    key_simple  = f'simple_{df}'
                    key_complex = f'complex_{df}'
                    if (key_simple  in diff_stim_data[uid] and
                        key_complex in diff_stim_data[uid]):
                        a = diff_stim_data[uid][key_simple].copy().pick('eeg')
                        b = diff_stim_data[uid][key_complex].copy().pick('eeg')
                        diff = combine_evoked([a, b], weights=[1, -1])
                        diff.comment = uid
                        taskdiff_stim_data[uid][df] = diff

            # resp raw: simple - complex (active only)
            if RUN_RESP_RAW and PARADIGM_REAL == 'active':
                key_simple  = 'simple_response'
                key_complex = 'complex_response'
                if (key_simple  in raw_resp_data[uid] and
                    key_complex in raw_resp_data[uid]):
                    a = raw_resp_data[uid][key_simple].copy().pick('eeg')
                    b = raw_resp_data[uid][key_complex].copy().pick('eeg')
                    diff = combine_evoked([a, b], weights=[1, -1])
                    diff.comment = uid
                    taskdiff_resp_data[uid]['response'] = diff

        # save taskdiff pkls
        if RUN_STIM_RAW:
            pivot_and_save(
                taskdiff_stim_data,
                stim_events,
                output_path_taskdiff,
                '{date}_{paradigm}_{count}p_taskdiff_stim_raw.pkl',
                '[taskdiff] stim raw',
            )

        if RUN_STIM_DIFF:
            pivot_and_save(
                taskdiff_stim_data,
                diff_events,
                output_path_taskdiff,
                '{date}_{paradigm}_{count}p_taskdiff_stim_diff.pkl',
                '[taskdiff] stim diff',
            )

        if RUN_RESP_RAW and PARADIGM_REAL == 'active':
            pivot_and_save(
                taskdiff_resp_data,
                ['response'],
                output_path_taskdiff_resp,
                '{date}_{paradigm}_{count}p_taskdiff_resp_{BASELINE_CORR}_{label_csd}_raw.pkl'.format(
                    date=date, paradigm=PARADIGM,
                    count=len(PARTICIPANTS),
                    BASELINE_CORR=BASELINE_CORR,
                    label_csd=label_csd,
                ),
                '[taskdiff] resp raw',
            )

        print('Task difference done.')